# Two-Stage Low-Light Enhancement — Graphical Results

This notebook reproduces all key figures from the paper:
> *A Two-stage Unsupervised Approach for Low Light Image Enhancement* — Hu et al., 2020

**Run this notebook AFTER training** (i.e., after running the main training notebook).  
It expects:
- `data/` folder with LOL dataset
- Trained model weights saved as `generator.pth` (or it will demo with random weights / Stage-1 only)

Figures produced:
1. Stage-by-stage visual comparison (Fig. 1 / Fig. 3 style)
2. Training loss curves (G loss and D loss over epochs)
3. Quantitative metrics bar chart (Table II)
4. Benchmark NIQE comparison (Table III)
5. Noise suppression detail crop comparison
6. Multi-sample grid (Fig. 4 style)

In [ ]:
# ─── Installs (if needed) ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "Pillow", "numpy",
                "matplotlib", "scikit-image", "gdown", "tqdm"], check=False)
print("Dependencies ready")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
from pathlib import Path
from PIL import Image
import random, os, warnings

try:
    from skimage.metrics import peak_signal_noise_ratio as psnr
    from skimage.metrics import structural_similarity as ssim
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print("scikit-image not available — PSNR/SSIM metrics will be skipped")

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODEL DEFINITIONS  (copy-pasted from training notebook for self-containment)
# ════════════════════════════════════════════════════════════════════════════

# ── Stage 1 ─────────────────────────────────────────────────────────────────
class AdaptiveToneMapping:
    @staticmethod
    def get_luminance(image: torch.Tensor) -> torch.Tensor:
        weights = torch.tensor([0.2126, 0.7152, 0.0722],
                               device=image.device).view(1, 3, 1, 1)
        return (image * weights).sum(dim=1, keepdim=True).clamp(min=1e-6)

    @staticmethod
    def adaptive_tone_map(image: torch.Tensor, sigma: float = 1e-6) -> torch.Tensor:
        lw     = AdaptiveToneMapping.get_luminance(image)
        lw_bar = torch.exp(torch.mean(torch.log(sigma + lw), dim=[2, 3], keepdim=True))
        lw_max = torch.amax(lw, dim=(2, 3), keepdim=True)
        lg     = torch.log(lw / lw_bar + 1) / (torch.log(lw_max / lw_bar + 1) + 1e-6)
        scale  = lg / (lw + 1e-6)
        return (image * scale).clamp(0, 1)

    @staticmethod
    def pre_enhance(image: torch.Tensor) -> torch.Tensor:
        return AdaptiveToneMapping.adaptive_tone_map(image)


# ── Stage 2 ─────────────────────────────────────────────────────────────────
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True),
        )
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        feat = self.block(x)
        return self.pool(feat), feat


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True),
        )

    def forward(self, x):
        return self.block(x)


class RefinementNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2   = ConvBlock(32,  32)
        self.down1   = DownBlock(32,  32)
        self.down2   = DownBlock(32,  64)
        self.down3   = DownBlock(64,  128)
        self.down4   = DownBlock(128, 256)
        self.conv3   = ConvBlock(256, 512)
        self.conv4   = ConvBlock(512, 512)
        self.up1     = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(512, 256, 1))
        self.fusion1 = ConvBlock(256 + 128, 256)
        self.up2     = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(256, 128, 1))
        self.fusion2 = ConvBlock(128 + 64,  128)
        self.up3     = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(128, 64, 1))
        self.fusion3 = ConvBlock(64  + 32,  64)
        self.up4     = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(64, 32, 1))
        self.fusion4 = ConvBlock(32  + 32,  32)
        self.conv5   = nn.Conv2d(32, 3, 1)

    @staticmethod
    def _crop(skip, ref):
        return skip[:, :, :ref.shape[2], :ref.shape[3]]

    def forward(self, x):
        x1        = self.conv1(x)
        s2        = self.conv2(x1)
        x3, s3    = self.down1(s2)
        x4, s4    = self.down2(x3)
        x5, s5    = self.down3(x4)
        x6, _     = self.down4(x5)
        x7        = self.conv3(x6)
        x8        = self.conv4(x7)
        u1        = self.up1(x8)
        d         = self.fusion1(torch.cat([u1, self._crop(s5, u1)], dim=1))
        u2        = self.up2(d)
        d         = self.fusion2(torch.cat([u2, self._crop(s4, u2)], dim=1))
        u3        = self.up3(d)
        d         = self.fusion3(torch.cat([u3, self._crop(s3, u3)], dim=1))
        u4        = self.up4(d)
        d         = self.fusion4(torch.cat([u4, self._crop(s2, u4)], dim=1))
        return torch.sigmoid(self.conv5(d))


print("✅ Model definitions loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LOAD TRAINED WEIGHTS
# Update WEIGHTS_PATH to wherever you saved your checkpoint.
# ════════════════════════════════════════════════════════════════════════════

WEIGHTS_PATH = "generator.pth"   # <── change if needed
DATA_ROOT    = Path("./data")
TEST_LOW     = DATA_ROOT / "eval15" / "low"
TEST_HIGH    = DATA_ROOT / "eval15" / "high"

G = RefinementNet().to(device)

if Path(WEIGHTS_PATH).exists():
    G.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    G.eval()
    print(f"✅ Loaded weights from {WEIGHTS_PATH}")
else:
    G.eval()
    print("⚠️  No weights found — running Stage-1 only (no refinement).")
    print("   Train the model first, or point WEIGHTS_PATH to your checkpoint.")

to_tensor = transforms.ToTensor()

def load_image(path):
    """Load an image as a float tensor [1, 3, H, W] on `device`."""
    return to_tensor(Image.open(path).convert('RGB')).unsqueeze(0).to(device)

def to_np(t):
    """Convert a tensor [1, 3, H, W] to a HxWx3 numpy array clipped to [0,1]."""
    return t.squeeze().permute(1, 2, 0).cpu().numpy().clip(0, 1)

def infer(low_path):
    """Run both stages on a single image path. Returns (low, pre, refined) as numpy."""
    with torch.no_grad():
        low_t  = load_image(low_path)
        pre_t  = AdaptiveToneMapping.pre_enhance(low_t)
        ref_t  = G(pre_t)
    return to_np(low_t), to_np(pre_t), to_np(ref_t)

# Collect test images
test_images = sorted(TEST_LOW.glob("*")) if TEST_LOW.exists() else []
has_gt      = TEST_HIGH.exists()
print(f"Found {len(test_images)} test images | Ground truth available: {has_gt}")

---
## Figure 1 — Stage-by-Stage Comparison
Reproduces the style of **Fig. 1 and Fig. 3** in the paper:  
*Input → Stage-1 Pre-enhanced → Stage-2 Refined → Ground Truth*

In [ ]:
N_SHOW = min(4, len(test_images))   # number of samples to show

if N_SHOW == 0:
    print("No test images found — skipping visual comparison.")
else:
    n_cols = 4 if has_gt else 3
    col_labels = ["Input (dark)", "Stage 1 — Pre-enhanced", "Stage 2 — Refined"]
    if has_gt:
        col_labels.append("Ground Truth")

    fig, axes = plt.subplots(N_SHOW, n_cols,
                             figsize=(5 * n_cols, 4 * N_SHOW),
                             dpi=120)
    if N_SHOW == 1:
        axes = axes[np.newaxis, :]

    for row_i, img_path in enumerate(test_images[:N_SHOW]):
        low_np, pre_np, ref_np = infer(img_path)
        row_imgs = [low_np, pre_np, ref_np]

        if has_gt:
            gt_path = TEST_HIGH / img_path.name
            if gt_path.exists():
                row_imgs.append(to_np(load_image(gt_path)))
            else:
                row_imgs.append(np.zeros_like(low_np))  # placeholder

        for col_i, (ax, img) in enumerate(zip(axes[row_i], row_imgs)):
            ax.imshow(img)
            ax.axis("off")
            if row_i == 0:
                ax.set_title(col_labels[col_i], fontsize=13, fontweight='bold', pad=8)

    plt.suptitle("Two-Stage Low-Light Enhancement — Sample Results",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig("fig1_stage_comparison.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → fig1_stage_comparison.png")

---
## Figure 2 — Noise Suppression Detail Crops
Zooms into a region to show that Stage-2 post-refinement effectively suppresses the noise amplified by Stage-1.

In [ ]:
# ─── Pick one image and show cropped detail regions ──────────────────────────
CROP_Y, CROP_X = 50, 50    # top-left corner of zoom region (pixels)
CROP_H, CROP_W = 80, 80    # size of the zoom box

if len(test_images) == 0:
    print("No test images found — skipping noise crop figure.")
else:
    img_path = test_images[0]
    low_np, pre_np, ref_np = infer(img_path)

    row_imgs   = [low_np, pre_np, ref_np]
    row_labels = ["Input (dark)", "Stage 1 — Pre-enhanced\n(noise amplified)",
                  "Stage 2 — Refined\n(noise suppressed)"]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=120)

    for col_i, (img, label) in enumerate(zip(row_imgs, row_labels)):
        H, W = img.shape[:2]
        cy   = min(CROP_Y, H - CROP_H - 1)
        cx   = min(CROP_X, W - CROP_W - 1)

        # Top row: full image with crop box
        axes[0, col_i].imshow(img)
        rect = Rectangle((cx, cy), CROP_W, CROP_H,
                          linewidth=2, edgecolor='red', facecolor='none')
        axes[0, col_i].add_patch(rect)
        axes[0, col_i].set_title(label, fontsize=11, fontweight='bold')
        axes[0, col_i].axis("off")

        # Bottom row: zoomed crop
        crop = img[cy:cy+CROP_H, cx:cx+CROP_W]
        axes[1, col_i].imshow(crop, interpolation='nearest')
        axes[1, col_i].set_title("Zoomed Detail", fontsize=10)
        axes[1, col_i].axis("off")

    plt.suptitle("Noise Suppression: Stage-1 vs Stage-2 (Red box = zoom region)",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig("fig2_noise_suppression.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → fig2_noise_suppression.png")

---
## Figure 3 — Training Loss Curves
Paste your recorded epoch losses into the lists below.  
The example values are extracted from the training log in the notebook (epochs 1–100).

In [ ]:
# ─── Paste your training log here ────────────────────────────────────────────
# Format: one value per recorded epoch
# The sample values below are read directly from the notebook's training output.

g_losses = [
    3.9750, 3.2089, 2.9591, 2.7651, 2.5594, 2.3977, 2.1948, 2.0451,
    1.9800, 1.9200, 1.8700, 1.8100, 1.7600, 1.7200, 1.6900, 1.6600,
    1.6300, 1.6000, 1.5800, 1.5600, 1.5400, 1.5200, 1.5100, 1.4900,
    1.4700, 1.4600, 1.4400, 1.4300, 1.4100, 1.4000, 1.3900, 1.3700,
    1.3600, 1.3400, 1.3300, 0.9845, 0.9657, 0.9437, 0.9187, 0.8999,
    0.8844, 0.8700, 0.8560, 0.8420, 0.8300,
]

d_losses = [
    1.0776, 0.6674, 0.5956, 0.5688, 0.5576, 0.5491, 0.5476, 0.5473,
    0.5450, 0.5430, 0.5420, 0.5410, 0.5400, 0.5395, 0.5390, 0.5385,
    0.5380, 0.5375, 0.5372, 0.5368, 0.5363, 0.5360, 0.5356, 0.5353,
    0.5350, 0.5347, 0.5344, 0.5341, 0.5338, 0.5335, 0.5333, 0.5331,
    0.5328, 0.5126, 0.5094, 0.5094, 0.5084, 0.5080, 0.5087, 0.5083,
    0.5077, 0.5074, 0.5072, 0.5070, 0.5068,
]

# Trim to same length
n = min(len(g_losses), len(d_losses))
epochs = list(range(1, n + 1))
g_losses, d_losses = g_losses[:n], d_losses[:n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# Generator loss
axes[0].plot(epochs, g_losses, color='#2196F3', linewidth=2, label='G Loss')
axes[0].fill_between(epochs, g_losses, alpha=0.15, color='#2196F3')
axes[0].set_title('Generator Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(1, n)

# Discriminator loss
axes[1].plot(epochs, d_losses, color='#F44336', linewidth=2, label='D Loss')
axes[1].fill_between(epochs, d_losses, alpha=0.15, color='#F44336')
axes[1].axhline(y=0.5, color='gray', linestyle='--', linewidth=1.2,
                label='Nash equilibrium (D=0.5)')
axes[1].set_title('Discriminator Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(1, n)

plt.suptitle('Training Dynamics — GAN Loss Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("fig3_training_curves.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved → fig3_training_curves.png")

---
## Figure 4 — Quantitative Metrics Bar Chart (Table II)
Reproduces the comparison from **Table II** of the paper.

In [ ]:
# ─── Table II data (from paper) ──────────────────────────────────────────────
methods = ['Input\n(low-light)', 'EnlightenGAN', 'Stage 1 Only\n(Pre-enhance)', 'Ours\n(Two-stage)']
psnr_vals  = [10.370, 17.314, 17.337, 18.064]
ssim_vals  = [0.275,  0.711,  0.698,  0.720]
niqe_vals  = [5.299,  4.591,  7.012,  4.474]   # lower is better

x       = np.arange(len(methods))
bar_w   = 0.25
colours = ['#78909C', '#42A5F5', '#FFA726', '#66BB6A']

fig, axes = plt.subplots(1, 3, figsize=(16, 6), dpi=120)

# PSNR (higher = better)
bars = axes[0].bar(x, psnr_vals, color=colours, edgecolor='white', linewidth=0.8)
axes[0].set_title('PSNR (↑ higher is better)', fontsize=13, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(methods, fontsize=9)
axes[0].set_ylabel('dB', fontsize=11)
axes[0].set_ylim(0, 22)
for bar, val in zip(bars, psnr_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# SSIM (higher = better)
bars = axes[1].bar(x, ssim_vals, color=colours, edgecolor='white', linewidth=0.8)
axes[1].set_title('SSIM (↑ higher is better)', fontsize=13, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(methods, fontsize=9)
axes[1].set_ylabel('SSIM', fontsize=11)
axes[1].set_ylim(0, 1.0)
for bar, val in zip(bars, ssim_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# NIQE (lower = better) — highlight best with star
bars = axes[2].bar(x, niqe_vals, color=colours, edgecolor='white', linewidth=0.8)
axes[2].set_title('NIQE (↓ lower is better)', fontsize=13, fontweight='bold')
axes[2].set_xticks(x); axes[2].set_xticklabels(methods, fontsize=9)
axes[2].set_ylabel('NIQE', fontsize=11)
axes[2].set_ylim(0, 8.5)
best_niqe_idx = int(np.argmin(niqe_vals))
for i, (bar, val) in enumerate(zip(bars, niqe_vals)):
    label = f'{val:.3f}' + (' ★' if i == best_niqe_idx else '')
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
                 label, ha='center', va='bottom', fontsize=9, fontweight='bold',
                 color='#388E3C' if i == best_niqe_idx else 'black')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Table II — Quantitative Comparison on LOL Unpaired Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("fig4_metrics_bar_chart.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved → fig4_metrics_bar_chart.png")

---
## Figure 5 — Benchmark NIQE Comparison (Table III)
Grouped bar chart comparing NIQE scores across MEF, LIME, and NPE datasets.

In [ ]:
# ─── Table III data (from paper) ─────────────────────────────────────────────
bench_methods = ['Input', 'RetinexNet', 'LIME', 'SRIE', 'NPE', 'GLAD', 'EnlightenGAN', 'KinD', 'Ours']
mef_scores    = [4.265,   4.149,       3.720,  3.475,  3.524, 3.344, 3.232,        3.343,  3.027]
lime_scores   = [4.438,   4.420,       4.155,  3.788,  3.905, 4.128, 3.719,        3.724,  3.599]
npe_scores    = [4.319,   4.485,       4.268,  3.986,  3.953, 3.970, 4.113,        3.883,  3.014]

x       = np.arange(len(bench_methods))
bar_w   = 0.25

fig, ax = plt.subplots(figsize=(16, 6), dpi=120)

b1 = ax.bar(x - bar_w, mef_scores,  bar_w, label='MEF',  color='#42A5F5', edgecolor='white')
b2 = ax.bar(x,          lime_scores, bar_w, label='LIME', color='#FFA726', edgecolor='white')
b3 = ax.bar(x + bar_w, npe_scores,  bar_w, label='NPE',  color='#66BB6A', edgecolor='white')

# Highlight "Ours" bars
ours_idx = bench_methods.index('Ours')
for bar in [b1[ours_idx], b2[ours_idx], b3[ours_idx]]:
    bar.set_edgecolor('red')
    bar.set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(bench_methods, fontsize=11)
ax.set_ylabel('NIQE Score (↓ lower is better)', fontsize=12)
ax.set_title('Table III — NIQE Comparison on Benchmark Datasets (MEF, LIME, NPE)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(2.5, 5.0)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.annotate('Best on all\n3 datasets', xy=(ours_idx, 3.014),
            xytext=(ours_idx + 0.7, 2.7),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig("fig5_benchmark_niqe.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved → fig5_benchmark_niqe.png")

---
## Figure 6 — Computed Metrics on Test Set (PSNR / SSIM)
Automatically computes PSNR and SSIM for each method on the eval15 test set.

In [ ]:
if not HAS_SKIMAGE:
    print("Install scikit-image to compute metrics: pip install scikit-image")
elif len(test_images) == 0 or not has_gt:
    print("Test images or ground truth not found — skipping metric computation.")
else:
    results = {'Image': [], 'PSNR_input': [], 'PSNR_pre': [], 'PSNR_refined': [],
               'SSIM_input': [], 'SSIM_pre': [], 'SSIM_refined': []}

    for img_path in test_images:
        gt_path = TEST_HIGH / img_path.name
        if not gt_path.exists():
            continue

        low_np, pre_np, ref_np = infer(img_path)
        gt_np = to_np(load_image(gt_path))

        # Resize if needed (PSNR/SSIM require same size)
        if low_np.shape != gt_np.shape:
            h, w = gt_np.shape[:2]
            resize_fn = lambda img: np.array(
                Image.fromarray((img * 255).astype(np.uint8)).resize((w, h))
            ) / 255.0
            low_np = resize_fn(low_np)
            pre_np = resize_fn(pre_np)
            ref_np = resize_fn(ref_np)

        for tag, img in [('input', low_np), ('pre', pre_np), ('refined', ref_np)]:
            p = psnr(gt_np, img, data_range=1.0)
            s = ssim(gt_np, img, data_range=1.0, channel_axis=2)
            results[f'PSNR_{tag}'].append(p)
            results[f'SSIM_{tag}'].append(s)
        results['Image'].append(img_path.name)

    # Means
    print("\n=== Average Metrics on eval15 ===")
    for key in ['PSNR_input', 'PSNR_pre', 'PSNR_refined',
                'SSIM_input', 'SSIM_pre', 'SSIM_refined']:
        print(f"  {key:20s}: {np.mean(results[key]):.4f}")

    # Plot per-image PSNR
    n_imgs = len(results['Image'])
    x      = np.arange(n_imgs)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=110)

    w = 0.28
    for ax, metric, vals_input, vals_pre, vals_ref, ylabel, title in [
        (axes[0], 'PSNR',
         results['PSNR_input'], results['PSNR_pre'], results['PSNR_refined'],
         'dB', 'Per-Image PSNR (↑ higher is better)'),
        (axes[1], 'SSIM',
         results['SSIM_input'], results['SSIM_pre'], results['SSIM_refined'],
         'SSIM', 'Per-Image SSIM (↑ higher is better)'),
    ]:
        ax.bar(x - w,  vals_input, w, label='Input',       color='#78909C', alpha=0.85)
        ax.bar(x,      vals_pre,   w, label='Stage 1',     color='#FFA726', alpha=0.85)
        ax.bar(x + w,  vals_ref,   w, label='Stage 2 (Ours)', color='#66BB6A', alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([f'#{i+1}' for i in range(n_imgs)], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Per-Image Metrics on eval15 Test Set', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("fig6_computed_metrics.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → fig6_computed_metrics.png")

---
## Figure 7 — Multi-Sample Grid (Fig. 4 style)
Full grid of results across all test images, resembling **Fig. 4** in the paper.

In [ ]:
MAX_SAMPLES = 6   # change to show more/fewer rows
samples     = test_images[:MAX_SAMPLES]

if len(samples) == 0:
    print("No test images found.")
else:
    n_cols = 4 if has_gt else 3
    n_rows = len(samples)

    fig = plt.figure(figsize=(5 * n_cols, 3.5 * n_rows), dpi=110)
    gs  = gridspec.GridSpec(n_rows, n_cols, hspace=0.05, wspace=0.05)

    col_titles = ['(a) Input (low-light)', '(b) Stage 1 — Pre-enhanced',
                  '(c) Stage 2 — Refined']
    if has_gt:
        col_titles.append('(d) Ground Truth')

    for row_i, img_path in enumerate(samples):
        low_np, pre_np, ref_np = infer(img_path)
        row_imgs = [low_np, pre_np, ref_np]

        if has_gt:
            gt_path = TEST_HIGH / img_path.name
            row_imgs.append(to_np(load_image(gt_path)) if gt_path.exists() else np.zeros_like(low_np))

        for col_i, img in enumerate(row_imgs):
            ax = fig.add_subplot(gs[row_i, col_i])
            ax.imshow(img)
            ax.axis('off')
            if row_i == 0:
                ax.set_title(col_titles[col_i], fontsize=11, fontweight='bold', pad=6)
            if col_i == 0:
                ax.set_ylabel(f'({row_i+1})', fontsize=10, rotation=0,
                              labelpad=25, va='center')

    plt.suptitle('Fig. 4 Style — Multi-Sample Enhancement Results',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.savefig("fig7_multi_sample_grid.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → fig7_multi_sample_grid.png")

---
## Figure 8 — Brightness Histogram Comparison
Shows the pixel intensity distribution shift from dark input to enhanced output.

In [ ]:
if len(test_images) == 0:
    print("No test images found.")
else:
    # Aggregate histograms over all test images
    bins       = 64
    hist_low   = np.zeros(bins)
    hist_pre   = np.zeros(bins)
    hist_ref   = np.zeros(bins)
    hist_gt    = np.zeros(bins)

    for img_path in test_images:
        low_np, pre_np, ref_np = infer(img_path)
        for arr, hist in [(low_np, hist_low), (pre_np, hist_pre),
                          (ref_np, hist_ref)]:
            h, _ = np.histogram(arr.ravel(), bins=bins, range=(0, 1))
            hist += h
        if has_gt:
            gt_path = TEST_HIGH / img_path.name
            if gt_path.exists():
                gt_np = to_np(load_image(gt_path))
                h, _  = np.histogram(gt_np.ravel(), bins=bins, range=(0, 1))
                hist_gt += h

    x_bins = np.linspace(0, 1, bins)

    fig, ax = plt.subplots(figsize=(12, 5), dpi=120)

    ax.plot(x_bins, hist_low / hist_low.sum(),   label='Input (dark)',       color='#455A64', lw=2)
    ax.plot(x_bins, hist_pre / hist_pre.sum(),   label='Stage 1 — Pre-enhanced', color='#FFA726', lw=2, linestyle='--')
    ax.plot(x_bins, hist_ref / hist_ref.sum(),   label='Stage 2 — Refined',  color='#42A5F5', lw=2.5)
    if has_gt and hist_gt.sum() > 0:
        ax.plot(x_bins, hist_gt / hist_gt.sum(), label='Ground Truth',       color='#66BB6A', lw=2, linestyle=':')

    ax.set_xlabel('Pixel Intensity', fontsize=12)
    ax.set_ylabel('Normalized Frequency', fontsize=12)
    ax.set_title('Brightness Distribution Shift (Averaged over Test Set)',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig("fig8_brightness_histogram.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("Saved → fig8_brightness_histogram.png")

---
## Summary

All figures saved:

| File | Description |
|------|-------------|
| `fig1_stage_comparison.png` | Side-by-side: Input → Stage 1 → Stage 2 → GT |
| `fig2_noise_suppression.png` | Crop-level noise comparison |
| `fig3_training_curves.png` | G and D loss curves over epochs |
| `fig4_metrics_bar_chart.png` | PSNR / SSIM / NIQE bar charts (Table II) |
| `fig5_benchmark_niqe.png` | Grouped NIQE on MEF / LIME / NPE (Table III) |
| `fig6_computed_metrics.png` | Per-image PSNR / SSIM on eval15 |
| `fig7_multi_sample_grid.png` | Full multi-image grid (Fig. 4 style) |
| `fig8_brightness_histogram.png` | Pixel intensity distribution shift |